# The Anatomy of AI · Lecture 6 · Detectors That Slide — the notebook

In the lecture a small detector, slid across a whole picture, found its pattern anywhere; stacked in layers, detectors
found edges, then curves, then car parts, then a car, all on one photo of a London cab. Here you rebuild it:

1. train **LeNet**, the first network of sliding detectors, and watch it shrug off a digit moved two squares;
2. slide a 3 × 3 edge detector over the cab **by hand**;
3. check one of InceptionV1's real first-layer detectors by hand against the network, to four decimals;
4. **max-pool** a map; measure how much of the photo each layer can see;
5. follow the cab to the curve, window, wheel, body and car detectors, and rank each against all 50,000 test photos;
6. find the photos that fire a detector hardest, and the network's answer.

Each section names the lecture plate it follows. Wherever the lecture gives a number, a `check(...)` line compares
yours with it and prints ✓; if a check fails, the notebook stops.

**Runs on:** free Colab with a T4 GPU (Runtime → Change runtime type → T4), in about ten minutes. It downloads MNIST
(11 MB), the model (27 MB) and the lecture's own copy of the 50,000 ImageNet test photos (two files of about 350 MB).

## 0. Setup
Installs (pinned), the plot style, and `check()`.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", "a GPU" if DEV == "cuda" else "the CPU")
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

### The lecture's own numbers
Every number below comes from the lecture's logs. The checks compare what you compute against these.

In [ ]:
LECTURE = json.loads(r'''{"lenet": {"params": 61706, "acc_runs": [0.9906, 0.9884, 0.987], "acc_site": 0.9887, "p30": 1.0, "shift": [0.9906, 0.9865, 0.9712, 0.9225, 0.7859], "templates_shift": [0.9206, 0.8655, 0.6652, 0.4469, 0.2367]}, "photo": {"index": 3111, "label": "cab", "numbers": 150528, "dot_150_60": [0.47, 0.48, 0.48]}, "params": 7066380, "macs": 1584881536, "edge": {"vertical_edge": {"at": [48, 41], "value": 3.003, "min": -2.779, "max": 3.003, "strong": 0.0644, "placements": 49284, "multiplies": 443556}, "horizontal_edge": {"at": [50, 32], "value": -3.196, "min": -3.196, "max": 3.168, "strong": 0.142, "placements": 49284, "multiplies": 443556}}, "conv2d0": {"shape": [64, 3, 7, 7], "channel": 2, "at": [25, 25], "by_hand": 4956.5928, "network": 4956.5923, "pos": 5569.51, "neg": -613.18, "bias": 0.2615, "pad3": 3633.7}, "pool": {"at": [24, 24], "patch": [[1214.0, 1180.5, 1081.5999755859375, 993.2999877929688], [4939.7001953125, 4956.60009765625, 4925.5, 4854.10009765625], [3240.0, 3259.60009765625, 3328.5, 3480.300048828125], [66.69999694824219, 90.19999694824219, 113.5999984741211, 167.6999969482422]], "pooled": [[4956.60009765625, 4925.5], [3259.60009765625, 3480.300048828125]]}, "layers": {"conv2d0": {"shape": [64, 112, 112], "zeros": 0.491, "rf": 7}, "conv2d1": {"shape": [64, 56, 56], "zeros": 0.35, "rf": 11}, "conv2d2": {"shape": [192, 56, 56], "zeros": 0.727, "rf": 19}, "mixed3a": {"shape": [256, 28, 28], "zeros": 0.405, "rf": 59}, "mixed3b": {"shape": [480, 28, 28], "zeros": 0.745, "rf": 91}, "mixed4a": {"shape": [508, 14, 14], "zeros": 0.67, "rf": 171}, "mixed4b": {"shape": [512, 14, 14], "zeros": 0.582, "rf": 214}, "mixed4c": {"shape": [512, 14, 14], "zeros": 0.645, "rf": 224}, "mixed4d": {"shape": [528, 14, 14], "zeros": 0.784, "rf": 224}, "mixed4e": {"shape": [832, 14, 14], "zeros": 0.726, "rf": 224}, "mixed5a": {"shape": [832, 7, 7], "zeros": 0.718, "rf": 224}, "mixed5b": {"shape": [1024, 7, 7], "zeros": 0.861, "rf": 224}}, "named": {"mixed3b:379": {"max": 595.98, "pct": 83.44, "peak": [9, 12], "label": "curve"}, "mixed3b:385": {"max": 743.77, "pct": 96.05, "peak": [9, 22], "label": "curve"}, "mixed4b:373": {"max": 978.5, "pct": 99.69, "peak": [7, 11], "label": "wheels"}, "mixed4b:237": {"max": 996.94, "pct": 99.87, "peak": [4, 11], "label": "windows"}, "mixed4b:491": {"max": 905.8, "pct": 99.96, "peak": [7, 3], "label": "car body"}, "mixed4c:447": {"max": 2414.36, "pct": 99.98, "peak": [6, 10], "label": "car"}, "mixed4d:161": {"max": 124.74, "pct": 70.72, "peak": [3, 7], "label": "jars"}, "mixed4e:55": {"max": 233.92, "pct": 99.0, "peak": [6, 5], "label": "cat / car fronts"}, "mixed5a:400": {"max": 10.73, "pct": 38.85, "peak": [6, 0], "label": "box camera"}, "conv2d0:2": {"max": 4956.59, "pct": 81.8, "peak": [25, 25], "label": "edge"}}, "top": {"mixed3b:379": [[35460, "wall clock", 1246.0], [36425, "magnetic compass", 1199.0], [32907, "odometer", 1175.0], [27182, "triumphal arch", 1162.0], [27389, "magnetic compass", 1158.0], [35506, "beer bottle", 1157.0]], "mixed4b:237": [[11276, "car wheel", 1289.0], [41872, "cab", 1267.0], [32218, "ambulance", 1252.0], [9097, "pickup", 1246.0], [7373, "minivan", 1232.0], [25468, "ambulance", 1226.0]], "mixed4b:373": [[49888, "ambulance", 1412.0], [8038, "ambulance", 1381.0], [49176, "minibus", 1346.0], [16612, "convertible", 1331.0], [1915, "ambulance", 1285.0], [25468, "ambulance", 1256.0]], "mixed4b:491": [[44804, "moving van", 1057.0], [37829, "golfcart", 997.0], [2326, "minibus", 996.0], [7843, "police van", 992.5], [1206, "minibus", 991.5], [48, "pickup", 988.5]], "mixed4c:447": [[25468, "ambulance", 2794.0], [26125, "police van", 2652.0], [5808, "limousine", 2536.0], [35787, "limousine", 2514.0], [8012, "fire engine", 2498.0], [38948, "trolleybus", 2486.0]], "mixed4d:161": [[17684, "pill bottle", 876.5], [42000, "pill bottle", 735.0], [49562, "water jug", 702.5], [17044, "corkscrew", 701.5], [15960, "hair spray", 691.5], [23146, "cocktail shaker", 678.5]], "mixed4e:55": [[38200, "Pomeranian", 487.5], [7405, "kit fox", 457.75], [23266, "tiger cat", 448.0], [38660, "tabby", 418.5], [22821, "tiger cat", 411.0], [29561, "red fox", 400.5]], "mixed5a:400": [[40660, "espresso", 302.25], [31314, "meat loaf", 281.25], [15051, "Polaroid camera", 263.75], [22956, "barn spider", 260.75], [11574, "pay-phone", 259.5], [22537, "gas pump", 256.5]]}, "top5": [["beach wagon", 0.9351], ["pickup", 0.0328], ["cab", 0.0131], ["minivan", 0.0074], ["fire engine", 0.0045]], "top1_all": 0.66416, "top5_all": 0.87356, "macs_mixed3b": 304267264, "total_numbers": 2840432, "total_zero_share": 0.61}''')
print(len(LECTURE['named']), 'named detectors,', len(LECTURE['layers']), 'layers in the lecture\'s record')

## 1. The problem with templates, and LeNet (Plates I–II)
Lecture 5's ten neurons read a digit with templates: one weight per square. Moved two squares to the right, the test
digits fall from 92.1% to 66.5% right. **LeNet** (LeCun and colleagues, 1989) instead slides small detectors across
the whole digit. LeNet-5: six detectors of 5 × 5 squares, a shrinking step, sixteen more detectors, another shrinking
step, then three ordinary layers. We train it the lecture's way: seed 0, eight passes over the 60,000 training digits,
batches of 128, the Adam optimiser.

In [ ]:
import gzip, requests
import torch.nn as nn
MIRROR = "https://ossci-datasets.s3.amazonaws.com/mnist/"
def fetch(name, offset):
    return np.frombuffer(gzip.decompress(requests.get(MIRROR + name, timeout=120).content), np.uint8, offset=offset)
Xtr = torch.tensor(fetch("train-images-idx3-ubyte.gz", 16).reshape(-1, 1, 28, 28)).float().div(255)
Ytr = torch.tensor(fetch("train-labels-idx1-ubyte.gz", 8).astype(np.int64))
Xte = torch.tensor(fetch("t10k-images-idx3-ubyte.gz", 16).reshape(-1, 1, 28, 28)).float().div(255)
Yte = torch.tensor(fetch("t10k-labels-idx1-ubyte.gz", 8).astype(np.int64))
print("downloaded MNIST:", len(Xtr), "training and", len(Xte), "test digits")

torch.manual_seed(0)
lenet = nn.Sequential(nn.Conv2d(1, 6, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),     # 6 sliding 5x5 detectors, then shrink
                      nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2),               # 16 more, reading the 6 maps
                      nn.Flatten(), nn.Linear(400, 120), nn.ReLU(), nn.Linear(120, 84), nn.ReLU(), nn.Linear(84, 10))
check("LeNet-5's weights", sum(p.numel() for p in lenet.parameters()), LECTURE["lenet"]["params"])
lenet = lenet.to(DEV)
opt = torch.optim.Adam(lenet.parameters(), 1e-3)
Xtr_d, Ytr_d, Xte_d, Yte_d = Xtr.to(DEV), Ytr.to(DEV), Xte.to(DEV), Yte.to(DEV)


def lenet_acc(x):
    with torch.no_grad():
        return float((lenet(x).argmax(1) == Yte_d).float().mean())


for p in range(8):
    order = torch.randperm(len(Xtr))
    for j in range(0, len(Xtr), 128):
        idx = order[j:j + 128].to(DEV)
        loss = F.cross_entropy(lenet(Xtr_d[idx]), Ytr_d[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"pass {p + 1}: {lenet_acc(Xte_d):.2%} of the test digits right")
runs = LECTURE["lenet"]["acc_runs"]
print(f"the lecture's runs ended at {min(runs):.2%} to {max(runs):.2%}")
check("LeNet after eight passes (the lecture's runs: 98.70-99.06%)", lenet_acc(Xte_d), LECTURE["lenet"]["acc_site"], tol=0.004)

Now Plate I's test: move every test digit 0–4 squares to the right (blank paper comes in from the left), with no
retraining. The templates' numbers are Lecture 5's.

In [ ]:
def shift(x, k):
    out = torch.zeros_like(x)
    out[..., k:] = x[..., :28 - k] if k else x
    return out

moved = [lenet_acc(shift(Xte_d, k)) for k in range(5)]
print("squares moved:   0      1      2      3      4")
print("templates:     " + "  ".join(f"{v:.1%}" for v in LECTURE["lenet"]["templates_shift"]))
print("LeNet:         " + "  ".join(f"{v:.1%}" for v in moved))
check("LeNet, digits moved 1 square", moved[1], LECTURE["lenet"]["shift"][1], tol=0.006)
check("LeNet, digits moved 2 squares", moved[2], LECTURE["lenet"]["shift"][2], tol=0.012)
check("moved 2 squares, LeNet loses", moved[0] - moved[2], LECTURE["lenet"]["shift"][0] - LECTURE["lenet"]["shift"][2], tol=0.012)

# Plate II: the six first-layer detectors, slid over Lecture 5's digit #30
with torch.no_grad():
    maps = lenet[1](lenet[0](Xte_d[30:31]))[0].cpu()
    p30 = lenet(Xte_d[30:31]).softmax(1)[0, 3].item()
check("LeNet's chance that digit #30 is a 3", p30, LECTURE["lenet"]["p30"], tol=0.001)
INKMAP = plt.matplotlib.colors.LinearSegmentedColormap.from_list("ink", [PAPER, INK])
w1 = lenet[0].weight.detach().cpu()[:, 0]
fig, axs = plt.subplots(2, 7, figsize=(10, 3.2))
axs[0, 0].imshow(Xte[30, 0], cmap=INKMAP); axs[0, 0].set_title("digit #30", fontsize=9)
axs[1, 0].axis("off"); axs[0, 0].axis("off")
for k in range(6):
    axs[0, k + 1].imshow(w1[k], cmap="gray"); axs[0, k + 1].set_title(f"detector {k}", fontsize=9)
    axs[1, k + 1].imshow(maps[k], cmap=INKMAP); axs[1, k + 1].set_title("its map", fontsize=9)
for a in axs.flat:
    a.axis("off")
plt.show()

## 2. A million photos, and one cab (Plate III)
ImageNet: 1.28 million photos of 1,000 kinds of things, labelled by people; another 50,000, 50 of each kind, are held
back for testing. **InceptionV1** (Szegedy and colleagues, Google, 2014) learned from them. We read the lecture's exact
copy of the 50,000 test photos, and the network through lucent's port, with its normalization fixed.

In [ ]:
# The photos: ImageNet's 50,000 validation photos, in the exact copy the course measured (a public 256-pixel copy,
# benjamin-paine/imagenet-1k-256x256 on Hugging Face: two parquet files of 25,000 photos, about 350 MB each, no key
# needed). Photo n here is photo n in the lecture. We read 100 photos at a time, only the ones we ask for.
import pyarrow.parquet as pq
import requests
from PIL import Image

SHARD = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "validation-{:05d}-of-00002.parquet")
_files = {}


def _shard(s):
    path = f"val-{s}.parquet"
    if not os.path.exists(path):
        t = time.time()
        with requests.get(SHARD.format(s), stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded photo file {s} ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t:.0f} s")
    if s not in _files:
        _files[s] = pq.ParquetFile(path)
    return _files[s]


def rows(ids):
    """{n: (jpeg bytes, class number 0..999)} for the photos with these numbers."""
    groups = {}
    for n in ids:
        groups.setdefault((n // 25000, (n % 25000) // 100), []).append(int(n))
    out = {}
    for (s, g), ns in sorted(groups.items()):
        t = _shard(s).read_row_group(g, columns=["image", "label"])
        ims, labs = t.column("image").to_pylist(), t.column("label").to_pylist()
        for n in ns:
            out[n] = (ims[n % 100]["bytes"], int(labs[n % 100]))
    return out


def decode(b, size=224):
    """JPEG bytes (256 px square) -> the centre 224 x 224, as a 3 x 224 x 224 tensor in [0, 1]."""
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    if (w, h) != (256, 256):
        s = 256 / min(w, h)
        im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
        w, h = im.size
    l, t = (w - size) // 2, (h - size) // 2
    im = im.crop((l, t, l + size, t + size))
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)

In [ ]:
# InceptionV1 (GoogLeNet, 2014), the network this lecture studies, through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()

import contextlib
from lucent.modelzoo import inceptionv1
with contextlib.redirect_stderr(io.StringIO()):            # keep the download's percentage bar off the page
    net = inceptionv1(pretrained=True).eval().to(DEV)
print("InceptionV1's weights (inception5h, 27 MB): ready")
for p in net.parameters():
    p.requires_grad_(False)

# Make sure the fix took: watch the normalization during one pass.
_M = importlib.import_module("lucent.modelzoo.inceptionv1.InceptionV1")
_seen, _orig = [], _M.F.local_response_norm
_M.F.local_response_norm = lambda x, size, alpha=1e-4, beta=0.75, k=1.0: (_seen.append((size, k)), _orig(x, size, alpha, beta, k))[1]
with torch.no_grad():
    net(torch.zeros(1, 3, 224, 224, device=DEV))
_M.F.local_response_norm = _orig
check("normalization size and k, every call", sorted(set(_seen)), [(10, 2)])


def prep(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


class Taps:
    """Look inside the network. record=[module names] keeps their outputs in taps[name];
    edit={name: fn} replaces a module's output with fn(output) on the way through."""
    def __init__(self, record=(), edit=None):
        mods = dict(net.named_modules())
        self.out, self.h = {}, []
        for name in record:
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, n=name: self.out.__setitem__(n, o)))
        for name, fn in (edit or {}).items():
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, fn=fn: fn(o)))

    def __getitem__(self, name):
        return self.out[name]

    def __enter__(self):
        return self

    def __exit__(self, *a):
        for h in self.h:
            h.remove()


# InceptionV1 has 1,008 outputs in its own order; PERM puts the 1,000 classes in the standard order (the lecture's
# perm.json, which also fixes two mislabelled outputs), and NAMES gives each class its name.
PERM = torch.tensor(json.loads("[449, 450, 442, 443, 444, 445, 446, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 494, 495, 496, 497, 498, 499, 500, 501, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 474, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 214, 215, 217, 216, 213, 218, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 226, 613, 614, 615, 616, 617, 618, 619, 620, 612, 422, 423, 424, 425, 427, 426, 428, 429, 430, 438, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 6, 22, 193, 14, 173, 99, 87, 69, 116, 198, 43, 89, 200, 98, 161, 132, 32, 180, 42, 18, 207, 181, 105, 77, 189, 20, 204, 63, 31, 66, 21, 25, 45, 170, 119, 210, 107, 126, 88, 145, 59, 160, 152, 72, 33, 91, 4, 27, 113, 123, 36, 156, 109, 158, 131, 26, 71, 56, 146, 144, 125, 176, 139, 134, 90, 2, 192, 154, 150, 94, 5, 19, 191, 196, 117, 141, 68, 93, 51, 208, 184, 97, 29, 171, 124, 128, 47, 64, 211, 112, 187, 114, 86, 115, 79, 130, 140, 110, 82, 17, 177, 149, 15, 3, 41, 127, 84, 143, 133, 60, 172, 179, 118, 168, 148, 70, 197, 50, 106, 49, 151, 46, 205, 102, 28, 58, 155, 136, 202, 34, 62, 1, 159, 67, 174, 55, 10, 95, 8, 11, 201, 85, 153, 30, 190, 76, 206, 61, 163, 103, 209, 74, 35, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 224, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 225, 657, 658, 188, 129, 164, 157, 13, 53, 183, 195, 101, 39, 80, 147, 78, 120, 167, 108, 162, 165, 81, 52, 9, 65, 57, 12, 121, 186, 48, 23, 182, 40, 212, 44, 16, 178, 38, 83, 104, 96, 185, 122, 73, 135, 137, 138, 203, 54, 100, 175, 142, 166, 37, 111, 92, 199, 75, 194, 24, 7, 169, 447, 451, 448, 452, 453, 456, 457, 454, 455, 547, 853, 896, 223, 345, 246, 230, 232, 677, 265, 264, 522, 688, 845, 752, 540, 847, 704, 767, 233, 907, 967, 341, 718, 916, 307, 705, 683, 518, 905, 258, 807, 908, 296, 353, 785, 909, 884, 266, 733, 991, 849, 777, 811, 933, 941, 254, 985, 835, 533, 839, 689, 252, 940, 805, 300, 706, 779, 538, 817, 716, 872, 719, 949, 851, 820, 580, 833, 887, 707, 267, 673, 591, 539, 239, 377, 836, 576, 568, 891, 749, 563, 548, 890, 929, 701, 241, 987, 342, 914, 827, 721, 902, 382, 762, 303, 335, 301, 801, 690, 695, 370, 713, 797, 979, 761, 996, 674, 698, 583, 543, 708, 243, 268, 376, 347, 910, 881, 297, 545, 778, 898, 298, 670, 756, 856, 865, 720, 313, 550, 959, 966, 523, 529, 315, 821, 667, 579, 715, 253, 897, 972, 834, 336, 799, 1000, 662, 512, 346, 262, 316, 879, 660, 808, 796, 305, 235, 279, 919, 995, 356, 309, 784, 261, 712, 934, 299, 256, 348, 671, 757, 280, 971, 567, 955, 275, 792, 276, 236, 337, 911, 227, 684, 725, 703, 517, 581, 895, 249, 375, 840, 505, 551, 750, 573, 350, 344, 554, 369, 787, 696, 730, 562, 802, 924, 293, 525, 980, 659, 592, 748, 269, 961, 963, 294, 560, 770, 773, 582, 918, 892, 814, 228, 374, 988, 371, 687, 238, 546, 270, 244, 867, 973, 894, 508, 536, 697, 532, 818, 917, 782, 977, 763, 338, 339, 781, 984, 598, 922, 946, 302, 717, 509, 661, 866, 875, 920, 880, 271, 251, 871, 829, 290, 272, 764, 686, 869, 277, 824, 854, 691, 852, 260, 255, 728, 511, 599, 284, 588, 585, 595, 755, 915, 552, 699, 354, 351, 520, 513, 333, 870, 937, 295, 506, 921, 826, 564, 584, 504, 759, 685, 352, 877, 942, 994, 306, 527, 257, 679, 843, 732, 842, 850, 883, 783, 789, 575, 926, 722, 281, 596, 931, 901, 888, 841, 565, 245, 983, 372, 693, 964, 731, 381, 378, 857, 923, 285, 855, 314, 788, 838, 566, 373, 769, 556, 702, 542, 534, 572, 846, 939, 862, 976, 273, 860, 571, 863, 537, 927, 286, 570, 965, 668, 578, 694, 219, 541, 310, 663, 997, 876, 519, 760, 753, 586, 952, 751, 938, 355, 809, 521, 962, 221, 729, 510, 587, 379, 589, 559, 800, 709, 832, 950, 259, 380, 868, 747, 590, 776, 943, 549, 726, 557, 507, 288, 278, 960, 222, 986, 577, 925, 822, 858, 515, 234, 676, 237, 600, 775, 274, 593, 804, 263, 680, 340, 530, 998, 724, 528, 516, 514, 287, 957, 311, 692, 247, 794, 526, 574, 535, 810, 681, 828, 837, 945, 569, 561, 531, 304, 250, 978, 675, 786, 944, 970, 989, 903, 758, 555, 308, 780, 664, 710, 312, 594, 700, 282, 711, 289, 283, 766, 825, 291, 242, 597, 678, 882, 349, 765, 727, 544, 220, 292, 334, 666, 874, 990, 969, 558, 790, 682, 343, 936, 665, 524, 928, 317, 231, 906, 669, 958, 819, 795, 772, 502, 899, 912, 904, 935, 831, 503, 672, 951, 815, 723, 248, 240, 714, 553, 930, 791, 932, 861, 774, 803, 754, 813, 844, 771, 793, 974, 968, 873, 768, 975, 993, 885, 734, 736, 737, 738, 739, 740, 741, 742, 743, 744, 735, 745, 746, 318, 229, 319, 320, 321, 322, 323, 324, 325, 326, 816, 999, 953, 864, 806, 948, 830, 900, 798, 947, 859, 823, 361, 992, 359, 365, 368, 366, 363, 364, 367, 360, 362, 954, 848, 982, 330, 357, 358, 331, 327, 328, 332, 886, 913, 956, 893, 878, 812, 981, 329, 889]"), device=DEV)
from torchvision.models import GoogLeNet_Weights
NAMES = GoogLeNet_Weights.IMAGENET1K_V1.meta["categories"]

In [ ]:
# Every ReLU in lucent's port passes gradients backwards in a special way meant for drawing (Lecture 8). For honest
# measurements (the receptive fields below) we want the plain gradient, so we swap in the ordinary ReLU.
for _m in net.modules():
    if type(_m).__name__ == "RedirectedReluLayer":
        _m.forward = lambda x: F.relu(x)
check("InceptionV1's weights", sum(p.numel() for p in net.parameters()), LECTURE["params"])

cab_jpg, cab_label = rows([3111])[3111]
cab = decode(cab_jpg)
check("test photo #3,111 is labelled", NAMES[cab_label], LECTURE["photo"]["label"])
check("numbers the network is given: 224 x 224 dots x 3 colours", cab.numel(), LECTURE["photo"]["numbers"])
dot = [round(float(v), 2) for v in cab[:, 150, 60]]
check("the dot at row 150, column 60: red, green, blue", dot, LECTURE["photo"]["dot_150_60"])
fig, ax = plt.subplots(figsize=(3.4, 3.4)); ax.imshow(cab.permute(1, 2, 0)); ax.axis("off")
ax.set_title("test photo #3,111", fontsize=10); plt.show()

## 3. A detector is a tiny template; slide it everywhere (Plates IV–V)
In grey (0 = black, 1 = white), a 3 × 3 template that wants dark on the left and bright on the right:

$$\begin{pmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{pmatrix}$$

Lay it on one 3 × 3 patch, multiply, add. Then lay the same nine weights on **every** patch, row by row: this is a
**convolution**, and its answers make a map of where the vertical edges are.

In [ ]:
grey = (0.299 * cab[0] + 0.587 * cab[1] + 0.114 * cab[2]).numpy()          # the lecture's grey
VERT = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)
HORZ = VERT.T.copy()


def slide(img, k):
    """Lay the 3x3 template on every 3x3 patch: 222 x 222 places, 9 multiplies each."""
    out = np.zeros((img.shape[0] - 2, img.shape[1] - 2), np.float32)
    for r in range(out.shape[0]):
        for c in range(out.shape[1]):
            out[r, c] = (img[r:r + 3, c:c + 3] * k).sum()
    return out


t = time.time()
vmap = slide(grey, VERT)
print(f"{vmap.size:,} placements, {vmap.size * 9:,} multiplies, by hand in {time.time() - t:.1f} s")
want = LECTURE["edge"]["vertical_edge"]
r, c = np.unravel_index(np.abs(vmap).argmax(), vmap.shape)
check("placements", vmap.size, want["placements"])
check("the strongest answer, place (row, column)", [int(r), int(c)], want["at"])
check("the strongest answer", vmap[r, c], want["value"], tol=0.001)
print("the patch there:\n", grey[r:r + 3, c:c + 3].round(2))
check("share of places with a strong answer (size over 0.5)", (np.abs(vmap) > 0.5).mean(), want["strong"], tol=0.0001)
check("a flat patch (top-left corner, white sky) answers", vmap[0, 0], 0.0)

fig, axs = plt.subplots(1, 2, figsize=(7.4, 3.6))
axs[0].imshow(grey, cmap="gray"); axs[0].set_title("the cab in grey", fontsize=10)
axs[0].add_patch(plt.Rectangle((c - 0.5, r - 0.5), 3, 3, fill=False, ec=INK, lw=1.2))
axs[1].imshow(np.abs(vmap), cmap=INKMAP); axs[1].set_title("vertical edges: ink = size of the answer", fontsize=10)
for a in axs:
    a.axis("off")
plt.show()

## 4. Sixty-four real detectors, and one checked by hand (Plates VI–VII)
InceptionV1's first layer, `conv2d0`, has 64 detectors, each 7 × 7 dots in three colours: 147 weights. Each draws its
own map (a **channel**), stepping 2 dots at a time (its **stride**), so 112 × 112 places.

The network does not read colours as 0–1: it was trained on dots from −117 to 138 (`prep` above). It also pads the
photo with blank dots so the map comes out 112 across: like TensorFlow, 2 on the top and left, 3 on the bottom and
right. Check channel 2 at the place where its map is strongest.

In [ ]:
conv0 = dict(net.named_modules())["conv2d0_pre_relu_conv"]
W0, b0 = conv0.weight.detach().cpu().numpy(), conv0.bias.detach().cpu().numpy()
check("conv2d0's weights: detectors x colours x rows x columns", list(W0.shape), LECTURE["conv2d0"]["shape"])

with torch.no_grad(), Taps(["conv2d0_pre_relu_conv", "conv2d0"]) as T1:
    net(prep(cab[None].to(DEV)))
pre = T1["conv2d0_pre_relu_conv"][0].cpu().numpy()                    # the network's own answers, before the gate
edge2 = T1["conv2d0"][0, 2].cpu().numpy()                             # channel 2's map, after the gate
ch = LECTURE["conv2d0"]["channel"]
r, c = np.unravel_index(pre[ch].argmax(), pre[ch].shape)
check("where channel 2's map is strongest", [int(r), int(c)], LECTURE["conv2d0"]["at"])

xin = prep(cab).numpy()
padded = np.pad(xin, ((0, 0), (2, 3), (2, 3)))                       # 2 blank dots top and left, 3 bottom and right
patch = padded[:, 2 * r:2 * r + 7, 2 * c:2 * c + 7]                   # the 7 x 7 x 3 dots under it (stride 2)
prods = patch * W0[ch]                                                # 147 multiplies
by_hand = prods.sum() + b0[ch]
print(f"positive products {prods[prods > 0].sum():,.2f}, negative {prods[prods < 0].sum():,.2f}, bias {b0[ch]:.4f}")
check("positive products", prods[prods > 0].sum(), LECTURE["conv2d0"]["pos"], tol=0.01)
check("negative products", prods[prods < 0].sum(), LECTURE["conv2d0"]["neg"], tol=0.01)
check("bias", b0[ch], LECTURE["conv2d0"]["bias"], tol=0.0001)
check("by hand", by_hand, LECTURE["conv2d0"]["by_hand"], tol=0.002)
check("the network's own answer", pre[ch, r, c], LECTURE["conv2d0"]["network"], tol=0.002)
check("by hand minus the network", by_hand - pre[ch, r, c], 0.0, tol=0.01)

Wn = W0.transpose(0, 2, 3, 1)
Wn = (Wn - Wn.min((1, 2, 3), keepdims=True)) / (np.ptp(Wn, (1, 2, 3), keepdims=True) + 1e-8)
fig, axs = plt.subplots(4, 16, figsize=(10, 2.8))
for k, a in enumerate(axs.flat):
    a.imshow(Wn[k], interpolation="nearest"); a.axis("off")
    if k == ch:
        a.set_title("2", fontsize=8)
fig.suptitle("conv2d0's 64 detectors, each 7 x 7 in colour (each stretched to its own range)", fontsize=10)
plt.show()

## 5. Keep the strongest (Plate IX)
Between layers the maps are shrunk by **max-pooling**: split a map into 2 × 2 blocks and keep the biggest number in
each. Here is a real 4 × 4 corner of channel 2's edge map, at its strongest place.

In [ ]:
r0, c0 = LECTURE["pool"]["at"]
corner = edge2[r0:r0 + 4, c0:c0 + 4]
print(corner.round(1))
pooled = corner.reshape(2, 2, 2, 2).max(axis=(1, 3))                  # the biggest of each 2 x 2 block
print("kept:", pooled.round(1).tolist())
check("the corner, square by square: largest difference", np.abs(corner - np.array(LECTURE["pool"]["patch"])).max(), 0, tol=0.1)
check("the four survivors: largest difference", np.abs(pooled - np.array(LECTURE["pool"]["pooled"])).max(), 0, tol=0.1)
check("the whole 112 x 112 map, pooled 2 x 2", list(F.max_pool2d(torch.tensor(edge2)[None], 2).shape[1:]), [56, 56])

## 6. Smaller grids, more of them (Plate X)
Each layer reads the maps of the layer below. The grids shrink while the detectors multiply. How much of the photo can
one square of each layer see, its **receptive field**? Measure it: start from a random photo, ask which dots can change
the centre square of a layer at all (the gradient, Lecture 7's tool, is nonzero), and measure their span.

In [ ]:
LAYERS = ["conv2d0", "conv2d1", "conv2d2", "mixed3a", "mixed3b", "mixed4a", "mixed4b", "mixed4c", "mixed4d",
          "mixed4e", "mixed5a", "mixed5b"]
with torch.no_grad(), Taps(LAYERS + ["softmax2_pre_activation_matmul"]) as TC:
    net(prep(cab[None].to(DEV)))
acts = {L: TC[L][0].cpu() for L in LAYERS}
cab_logits = TC["softmax2_pre_activation_matmul"][0, PERM].cpu()

rf = {}
torch.manual_seed(0)
for L in LAYERS:
    x = torch.rand(1, 3, 224, 224, device=DEV, requires_grad=True)
    with Taps([L]) as TL:
        net(prep(x))
    a = TL[L]
    H, Wd = a.shape[-2:]
    a[0, :, H // 2, Wd // 2].sum().backward()
    ys, xs = np.where(x.grad[0].abs().sum(0).cpu().numpy() > 0)
    rf[L] = int(ys.max() - ys.min() + 1)
print(f"{'layer':9s} {'detectors':>9s} {'grid':>7s} {'sees':>6s}")
for L in LAYERS:
    C, H, Wd = acts[L].shape
    print(f"{L:9s} {C:9d} {H:4d}x{Wd:<3d} {rf[L]:4d} px")
    check(f"{L}: detectors, grid", list(acts[L].shape), LECTURE["layers"][L]["shape"])
    check(f"{L}: receptive field", rf[L], LECTURE["layers"][L]["rf"])

macs = []
def count(mod, inp, out):
    if isinstance(mod, nn.Conv2d):
        macs.append(out.shape[1] * out.shape[2] * out.shape[3] * mod.kernel_size[0] * mod.kernel_size[1] * mod.in_channels // mod.groups)
    elif isinstance(mod, nn.Linear):
        macs.append(mod.in_features * mod.out_features)
hooks = [m.register_forward_hook(count) for m in net.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
with torch.no_grad():
    net(prep(cab[None].to(DEV)))
for h in hooks:
    h.remove()
check("multiply-adds for one photo", sum(macs), LECTURE["macs"])

## 7. All 50,000 test photos, once
To say how strongly a detector fires on the cab *compared with every other photo* (its **percentile**), to find the
photos that fire it hardest, and to score the network, we need one pass over all 50,000 test photos. For each photo
we keep each named detector's strongest answer anywhere in it, where that was, and the network's top five answers.
This is the slow part: about four minutes on a T4.

In [ ]:
NAMED = {"conv2d0:2": "edge", "mixed3b:379": "curve", "mixed3b:385": "curve", "mixed4b:237": "windows",
         "mixed4b:373": "wheels", "mixed4b:491": "car body", "mixed4c:447": "car", "mixed4d:161": "jars",
         "mixed4e:55": "cat / car fronts", "mixed5a:400": "box camera"}
_shard(0), _shard(1)                                   # the two photo files, downloaded once


class Groups(torch.utils.data.Dataset):
    """The 50,000 photos in groups of 100, read and cut to 224 x 224 by worker processes."""
    def __len__(self):
        return 500

    def __getitem__(self, k):
        s, g = divmod(k, 250)
        t = pq.ParquetFile(f"val-{s}.parquet").read_row_group(g, columns=["image", "label"])
        return (torch.stack([decode(im["bytes"]) for im in t.column("image").to_pylist()]),
                torch.tensor(t.column("label").to_pylist()))


loader = torch.utils.data.DataLoader(Groups(), batch_size=2, num_workers=2,
                                     collate_fn=lambda b: (torch.cat([x for x, _ in b]), torch.cat([y for _, y in b])))
peak = {k: [] for k in NAMED}
where = {k: [] for k in NAMED}
top5, labels = [], []
t = time.time()
with torch.no_grad():
    for xb, yb in loader:
        with Taps(sorted({k.split(":")[0] for k in NAMED}) + ["softmax2_pre_activation_matmul"]) as TT:
            net(prep(xb.to(DEV)))
        for k in NAMED:
            L, c = k.split(":")
            m = TT[L][:, int(c)].flatten(1)
            v, i = m.max(1)
            peak[k].append(v.cpu()); where[k].append(i.cpu())
        top5.append(TT["softmax2_pre_activation_matmul"][:, PERM].topk(5, 1).indices.cpu())
        labels.append(yb)
peak = {k: torch.cat(v).numpy() for k, v in peak.items()}
where = {k: torch.cat(v).numpy() for k, v in where.items()}
top5, labels = torch.cat(top5).numpy(), torch.cat(labels).numpy()
print(f"{len(labels):,} photos in {time.time() - t:.0f} s")
check("photos", len(labels), 50000)

## 8. Edges, curves, parts, a car (Plates XI–XIV)
Back to the cab. For each named detector: its strongest answer on the cab, where (row, column of its grid), and its
percentile: the share of the 50,000 test photos whose own strongest answer is lower.

In [ ]:
print(f"{'detector':12s} {'what':16s} {'on the cab':>10s} {'where':>9s} {'percentile':>11s}")
for k in NAMED:
    L, c = k.split(":")
    m = acts[L][int(c)]
    v = float(m.max())
    yx = [int(t) for t in np.unravel_index(int(m.argmax()), m.shape)]
    pct = 100 * float((peak[k] < v).mean())
    want = LECTURE["named"][k]
    print(f"{k:12s} {NAMED[k]:16s} {v:10.1f} {str(yx):>9s} {pct:10.2f}%")
    check(f"{k} on the cab", v, want["max"], tol=0.02 * max(1, want["max"] / 100))
    check(f"{k}: where", yx, want["peak"])
    check(f"{k}: percentile among 50,000", pct, want["pct"], tol=0.1)

def overlay(ax, m, title):
    ax.imshow(cab.permute(1, 2, 0)); ax.imshow(np.sqrt(m / m.max()), cmap=INKMAP, alpha=0.8, extent=(0, 224, 224, 0))
    ax.set_title(title, fontsize=9); ax.axis("off")

fig, axs = plt.subplots(1, 6, figsize=(12, 2.4))
for ax, k in zip(axs, ["conv2d0:2", "mixed3b:379", "mixed4b:237", "mixed4b:373", "mixed4b:491", "mixed4c:447"]):
    L, c = k.split(":")
    overlay(ax, acts[L][int(c)].numpy(), f"{k}\n{NAMED[k]}")
plt.show()

## 9. The photos that fire it hardest (Plate XV)
Zeiler and Fergus (2013): rank all 50,000 photos by a detector's strongest answer. Below are the top four for the car
and the "pointed faces" detectors, each cropped around the square where it fired most. (The jar detector's photos are
checked but not shown: its top two are prescription bottles whose labels carry a person's name.)

In [ ]:
for k in ("mixed4c:447", "mixed4d:161", "mixed4e:55"):
    best = np.argsort(-peak[k])[:6]
    names = [NAMES[int(labels[i])] for i in best]
    print(f"{k}: " + ", ".join(f"#{i} {n}" for i, n in zip(best, names)))
    want = LECTURE["top"][k]
    check(f"{k}: its four strongest photos", [int(i) for i in best[:4]], [w[0] for w in want[:4]])
    check(f"{k}: the strongest answer of all", peak[k][best[0]], want[0][2], tol=0.002 * want[0][2])

fig, axs = plt.subplots(2, 4, figsize=(8, 4.2))
for row, k in zip(axs, ("mixed4c:447", "mixed4e:55")):
    L = k.split(":")[0]
    side = acts[L].shape[-1]
    for ax, i in zip(row, np.argsort(-peak[k])[:4]):
        im = decode(rows([int(i)])[int(i)][0])
        y, x = divmod(int(where[k][i]), side)
        cy, cx = int((y + 0.5) * 224 / side), int((x + 0.5) * 224 / side)
        y0, x0 = min(max(cy - 56, 0), 112), min(max(cx - 56, 0), 112)
        ax.imshow(im[:, y0:y0 + 112, x0:x0 + 112].permute(1, 2, 0)); ax.axis("off")
        ax.set_title(NAMES[int(labels[i])], fontsize=9)
    row[0].text(-20, 56, k, rotation=90, va="center", ha="right", fontsize=9)
plt.show()

## 10. The answer (Plate XVI)
At the end: 1,000 scores, one per kind of thing, turned into chances (Lecture 5's softmax).

In [ ]:
p = cab_logits.softmax(0)
best = p.topk(5)
for v, i in zip(best.values, best.indices):
    print(f"{NAMES[int(i)]:14s} {float(v):7.2%}")
for (name, want), v, i in zip(LECTURE["top5"], best.values, best.indices):
    check("the cab: next answer", NAMES[int(i)], name)
    check(f"the cab: chance of {name}", float(v), want, tol=0.0005)
check("on all 50,000 photos, the top answer is right", (top5[:, 0] == labels).mean(), LECTURE["top1_all"], tol=0.0003)
check("the right answer is in the top five", (top5 == labels[:, None]).any(1).mean(), LECTURE["top5_all"], tol=0.0003)

## 11. Most of it is silent (Plate XVII)
Every detector ends with Lecture 1's gate: below zero, its answer is exactly 0. Count the zeros, layer by layer.

In [ ]:
total = sum(a.numel() for a in acts.values())
zeros = sum(int((a == 0).sum()) for a in acts.values())
for L in LAYERS:
    z = float((acts[L] == 0).float().mean())
    check(f"{L}: share of numbers exactly 0", z, LECTURE["layers"][L]["zeros"], tol=0.0006)
check("all the numbers the cab becomes", total, LECTURE["total_numbers"])
check("share exactly 0", zeros / total, LECTURE["total_zero_share"], tol=0.005)
fig, ax = plt.subplots(figsize=(7, 2.4))
ax.bar(range(12), [float((acts[L] == 0).float().mean()) for L in LAYERS], color=PAPER, edgecolor=INK, hatch="///")
ax.set_xticks(range(12)); ax.set_xticklabels([L.replace("mixed", "") for L in LAYERS], fontsize=8)
ax.set_ylabel("share exactly 0"); ax.set_ylim(0, 1); ax.set_title("most of it is silent", fontsize=10)
plt.show()

## Your turn
1. **The horizontal detector.** Slide `HORZ` (the vertical one turned on its side) over the grey cab. Where is its
   strongest answer, and what share of places answer strongly? *Hint: `slide(grey, HORZ)`.*
2. **The wrong padding.** Pad 3 blank dots on every side instead of 2 and 3, and redo the by-hand check of channel 2.
   *Hint: `np.pad(xin, ((0, 0), (3, 3), (3, 3)))`, and the patch starts at `2 * r`.*
3. **The windows detector's favourite photos.** Which four photos fire mixed4b:237 hardest? *Hint: `peak["mixed4b:237"]`.*
4. **The other curve.** mixed3b:385 is a second curve detector. How hard does it fire on the cab, where, and above
   what share of photos? *Hint: it is already in `NAMED`.*
5. **Count the work.** How many multiply-adds does the mixed3b block do on one photo? *Hint: hook only the modules
   whose names start with `mixed3b_`.*

In [ ]:
# Your turn 1
# TODO

In [ ]:
# Your turn 2
# TODO

In [ ]:
# Your turn 3
# TODO

In [ ]:
# Your turn 4
# TODO

In [ ]:
# Your turn 5
# TODO

## Solutions

In [ ]:
# 1. The horizontal detector
hmap = slide(grey, HORZ)
r, c = np.unravel_index(np.abs(hmap).argmax(), hmap.shape)
want = LECTURE["edge"]["horizontal_edge"]
check("horizontal: strongest place", [int(r), int(c)], want["at"])
check("horizontal: strongest answer", hmap[r, c], want["value"], tol=0.001)
check("horizontal: share answering strongly", (np.abs(hmap) > 0.5).mean(), want["strong"], tol=0.0001)

In [ ]:
# 2. The wrong padding
r, c = LECTURE["conv2d0"]["at"]
wrong = np.pad(xin, ((0, 0), (3, 3), (3, 3)))[:, 2 * r:2 * r + 7, 2 * c:2 * c + 7]
v = (wrong * W0[ch]).sum() + b0[ch]
print(f"pad 3 all round: {v:,.2f}, against the network's {pre[ch, r, c]:,.1f}")
# The lecture first printed 3,633.7, from a GPU convolution in reduced precision; the exact sum, as here, is 3,634.0.
check("pad 3 all round gives", v, LECTURE["conv2d0"]["pad3"], tol=0.5)

In [ ]:
# 3. The windows detector's favourite photos
best = np.argsort(-peak["mixed4b:237"])[:4]
print([(int(i), NAMES[int(labels[i])]) for i in best])
check("mixed4b:237: its four strongest photos", [int(i) for i in best], [w[0] for w in LECTURE["top"]["mixed4b:237"][:4]])

In [ ]:
# 4. The other curve
m = acts["mixed3b"][385]
v, yx = float(m.max()), [int(t) for t in np.unravel_index(int(m.argmax()), m.shape)]
pct = 100 * float((peak["mixed3b:385"] < v).mean())
print(f"mixed3b:385 on the cab: {v:.1f} at {yx}, above {pct:.2f}% of photos")
check("mixed3b:385 on the cab", v, LECTURE["named"]["mixed3b:385"]["max"], tol=0.5)
check("mixed3b:385: where", yx, LECTURE["named"]["mixed3b:385"]["peak"])
check("mixed3b:385: percentile", pct, LECTURE["named"]["mixed3b:385"]["pct"], tol=0.06)

In [ ]:
# 5. Count the work
macs = []
hooks = [m.register_forward_hook(count) for n, m in net.named_modules()
         if n.startswith("mixed3b_") and isinstance(m, (nn.Conv2d, nn.Linear))]
with torch.no_grad():
    net(prep(cab[None].to(DEV)))
for h in hooks:
    h.remove()
check("multiply-adds in mixed3b", sum(macs), LECTURE["macs_mixed3b"])
print(f"notebook ran in {time.time() - T0:.0f} s")